In [1]:
import random
import numpy as np
from deap import base, creator, tools, algorithms
import pandas as pd
import scipy.stats as stats
import joblib
from sklearn.ensemble import RandomForestRegressor as RFR
from sklearn.feature_selection import RFE
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

## Exploration

In [2]:
model_optimalsameTR= joblib.load("model_optimalsameTR.pkl")
model_optimalsameTs = joblib.load("model_optimalsameTs.pkl")

In [3]:
# 定义适应度函数类
creator.create("FitnessMulti", base.Fitness, weights=(1.0, 1.0))
# 定义个体类
creator.create("Individual", list, fitness=creator.FitnessMulti)
toolbox = base.Toolbox()

In [4]:
def uniform(min, max):
    return random.uniform(min, max)
   
element_names = ["C", "Si", "Mn", "P", "S", "Ni", "Cr", "Mo", "W", "Cu", "B", "Al", "Ti",
                 "N", "V", "Nb", "Zr", "Nb+Ta", "Fe","Co", "Sn", "ρ(g/㎤)", "R(Å)", "∆R", "χ",
                 "∆χ", "VEC", "∆VEC", "TeWQ(℃)", "TiWQ(h)", "TeFC(℃)", "TiFC(h)","TeOQ(℃)", 
                 "TiOQ(h)", "TeAC(℃)", "TiAC(h)", "TeAr(℃)", "TiAr(h)","Rockwell hardness(HRC)", 
                 "Austenite grain size number","and", "arc", "as","basic", "cast", "cconverter","cold",
                 "consumable","converter","drawn", "electric", "extruded", "forged","frequency","furancy",
                 "high","hot", "induction","ld", "melting","pierced", "rolled", "rotary","vacuum", "Test Temparature(℃)", 
                 "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"] 
element_range = [(0.039,0.43),(0.77,1.2),(0.056,1.71),(0.002,0.023),(0.0008,0.026),(0,75.35),(0.04,25.6),(0,3.74),(0,2.48),(0,2.92),(0,0.0059),(0,3.07),(0,2.45),(0,0.28),(0,0.28),(0,0.88),(0,0.01),(0,5),(0,14.4515),(0,28.88),(0,0.027),
                 (5,9),(1,2),(0,1),(1,2),(0,1),(5,10),(1,3),[930,930],[6,6],[600,600],[2,2],[0,0],[0,0],[635,635],[6,6],[0,0],[0,0],(0,100)
                ,(0,10),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),(0,1),
                 (0,1),(0,1),(0,1),(0,1),(0,1000),(0,1000),(0,1000),(0,100),(0,100)]
toolbox.register("individual", tools.initIterate, creator.Individual,
                 lambda: [uniform(min, max) for min, max in element_range])
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [5]:
def evaluate(individual):
    selected_features = ["Si", "Mn", "Mo", "N", "Test Temparature(℃)", "0.2% Proof Stress(MPa)", "Test stress(MPa)", "Elongation(%)", "Reduction of Area(%)"]
    selected_values = []
    for feature in selected_features:
        index = element_names.index(feature)
        value = individual[index]
        selected_values.append(value)

    x_matrix = np.array(selected_values).reshape(1, -1)

    val_TR = model_optimalsameTR.predict(x_matrix)[0]
    val_TS = model_optimalsameTs.predict(x_matrix)[0]
    return val_TR, val_TS


In [6]:
def mutate(individual, min, max, indpb):
    for i in range(len(individual)):
        if random.random() < indpb:
            individual[i] = uniform(min[i], max[i])
    return individual,
min_values = [min_value for min_value, _ in element_range]
max_values = [max_value for _, max_value in element_range]

toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", mutate, min=min_values, max=max_values, indpb=0.2)
toolbox.register("select", tools.selNSGA2)
toolbox.register("evaluate", evaluate)

In [7]:
pop_size = 200
n_gen = 1000
CXPB = 0.8
MUTPB = 0.05

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	200   
1  	179   
2  	173   
3  	167   
4  	178   
5  	170   
6  	168   
7  	166   
8  	179   
9  	168   
10 	175   
11 	166   
12 	177   
13 	174   
14 	165   
15 	165   
16 	170   
17 	170   
18 	170   
19 	167   
20 	177   
21 	170   
22 	169   
23 	170   
24 	167   
25 	173   
26 	165   
27 	169   
28 	180   
29 	173   
30 	171   
31 	165   
32 	165   
33 	166   
34 	180   
35 	171   
36 	170   
37 	161   
38 	163   
39 	164   
40 	170   
41 	169   
42 	167   
43 	177   
44 	173   
45 	171   
46 	176   
47 	175   
48 	178   
49 	174   
50 	178   
51 	171   
52 	178   
53 	168   
54 	178   
55 	171   
56 	165   
57 	179   
58 	167   
59 	168   
60 	170   
61 	179   
62 	167   
63 	174   
64 	168   
65 	172   
66 	172   
67 	174   
68 	178   
69 	174   
70 	175   
71 	177   
72 	169   
73 	175   
74 	169   
75 	176   
76 	164   
77 	177   
78 	161   
79 	173   
80 	162   
81 	164   
82 	168   
83 	161   
84 	167   
85 	171   
86 	183   
87 	173   
88 	177   
89 	170   

744	171   
745	169   
746	171   
747	176   
748	168   
749	168   
750	172   
751	166   
752	170   
753	161   
754	172   
755	170   
756	159   
757	169   
758	178   
759	166   
760	181   
761	167   
762	171   
763	168   
764	167   
765	172   
766	163   
767	167   
768	166   
769	176   
770	180   
771	169   
772	176   
773	169   
774	176   
775	169   
776	166   
777	177   
778	175   
779	178   
780	166   
781	175   
782	164   
783	177   
784	171   
785	165   
786	174   
787	169   
788	174   
789	172   
790	170   
791	165   
792	160   
793	168   
794	170   
795	180   
796	166   
797	176   
798	169   
799	176   
800	174   
801	166   
802	170   
803	168   
804	171   
805	180   
806	160   
807	173   
808	169   
809	178   
810	171   
811	168   
812	169   
813	173   
814	171   
815	175   
816	159   
817	171   
818	160   
819	171   
820	171   
821	173   
822	168   
823	173   
824	168   
825	164   
826	165   
827	155   
828	178   
829	175   
830	165   
831	168   
832	170   
833	162   
834	166   

In [8]:
# 将帕累托前沿数据保存到Excel文件
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df = pd.DataFrame(pareto_solutions, columns=column_names)
df.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B探索.xlsx', index=False)

In [9]:
pop_size = 200
n_gen = 1000
CXPB = 0.9
MUTPB = 0.05

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	200   
1  	188   
2  	189   
3  	190   
4  	194   
5  	189   
6  	189   
7  	191   
8  	189   
9  	192   
10 	179   
11 	191   
12 	189   
13 	198   
14 	187   
15 	194   
16 	189   
17 	192   
18 	191   
19 	190   
20 	190   
21 	186   
22 	192   
23 	187   
24 	191   
25 	188   
26 	190   
27 	187   
28 	189   
29 	191   
30 	186   
31 	193   
32 	186   
33 	187   
34 	192   
35 	190   
36 	190   
37 	191   
38 	189   
39 	194   
40 	191   
41 	193   
42 	192   
43 	190   
44 	189   
45 	190   
46 	188   
47 	193   
48 	191   
49 	186   
50 	192   
51 	188   
52 	192   
53 	194   
54 	194   
55 	192   
56 	196   
57 	193   
58 	193   
59 	189   
60 	187   
61 	194   
62 	196   
63 	184   
64 	185   
65 	189   
66 	192   
67 	191   
68 	192   
69 	194   
70 	186   
71 	189   
72 	194   
73 	190   
74 	190   
75 	187   
76 	189   
77 	183   
78 	192   
79 	192   
80 	191   
81 	189   
82 	192   
83 	186   
84 	190   
85 	187   
86 	188   
87 	189   
88 	190   
89 	187   

745	190   
746	185   
747	185   
748	184   
749	196   
750	190   
751	190   
752	188   
753	191   
754	192   
755	191   
756	191   
757	193   
758	193   
759	191   
760	188   
761	192   
762	191   
763	183   
764	190   
765	182   
766	191   
767	193   
768	188   
769	190   
770	191   
771	184   
772	189   
773	187   
774	191   
775	193   
776	189   
777	190   
778	193   
779	188   
780	189   
781	187   
782	191   
783	187   
784	191   
785	191   
786	192   
787	189   
788	182   
789	191   
790	190   
791	194   
792	197   
793	188   
794	193   
795	192   
796	192   
797	188   
798	190   
799	185   
800	196   
801	186   
802	192   
803	187   
804	193   
805	190   
806	190   
807	190   
808	191   
809	191   
810	186   
811	190   
812	191   
813	189   
814	195   
815	191   
816	189   
817	192   
818	191   
819	190   
820	190   
821	190   
822	188   
823	190   
824	190   
825	191   
826	193   
827	190   
828	189   
829	191   
830	190   
831	186   
832	195   
833	190   
834	191   
835	191   

In [10]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B探索.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B探索.xlsx', index=False)

In [11]:
pop_size = 200
n_gen = 1000
CXPB = 0.9
MUTPB = 0.1

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	200   
1  	200   
2  	200   
3  	200   
4  	200   
5  	200   
6  	200   
7  	200   
8  	200   
9  	200   
10 	200   
11 	200   
12 	200   
13 	200   
14 	200   
15 	200   
16 	200   
17 	200   
18 	200   
19 	200   
20 	200   
21 	200   
22 	200   
23 	200   
24 	200   
25 	200   
26 	200   
27 	200   
28 	200   
29 	200   
30 	200   
31 	200   
32 	200   
33 	200   
34 	200   
35 	200   
36 	200   
37 	200   
38 	200   
39 	200   
40 	200   
41 	200   
42 	200   
43 	200   
44 	200   
45 	200   
46 	200   
47 	200   
48 	200   
49 	200   
50 	200   
51 	200   
52 	200   
53 	200   
54 	200   
55 	200   
56 	200   
57 	200   
58 	200   
59 	200   
60 	200   
61 	200   
62 	200   
63 	200   
64 	200   
65 	200   
66 	200   
67 	200   
68 	200   
69 	200   
70 	200   
71 	200   
72 	200   
73 	200   
74 	200   
75 	200   
76 	200   
77 	200   
78 	200   
79 	200   
80 	200   
81 	200   
82 	200   
83 	200   
84 	200   
85 	200   
86 	200   
87 	200   
88 	200   
89 	200   

744	200   
745	200   
746	200   
747	200   
748	200   
749	200   
750	200   
751	200   
752	200   
753	200   
754	200   
755	200   
756	200   
757	200   
758	200   
759	200   
760	200   
761	200   
762	200   
763	200   
764	200   
765	200   
766	200   
767	200   
768	200   
769	200   
770	200   
771	200   
772	200   
773	200   
774	200   
775	200   
776	200   
777	200   
778	200   
779	200   
780	200   
781	200   
782	200   
783	200   
784	200   
785	200   
786	200   
787	200   
788	200   
789	200   
790	200   
791	200   
792	200   
793	200   
794	200   
795	200   
796	200   
797	200   
798	200   
799	200   
800	200   
801	200   
802	200   
803	200   
804	200   
805	200   
806	200   
807	200   
808	200   
809	200   
810	200   
811	200   
812	200   
813	200   
814	200   
815	200   
816	200   
817	200   
818	200   
819	200   
820	200   
821	200   
822	200   
823	200   
824	200   
825	200   
826	200   
827	200   
828	200   
829	200   
830	200   
831	200   
832	200   
833	200   
834	200   

In [12]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B探索.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B探索.xlsx', index=False)

In [13]:
pop_size = 200
n_gen = 1000
CXPB = 0.8
MUTPB = 0.1

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	200   
1  	175   
2  	186   
3  	177   
4  	183   
5  	177   
6  	185   
7  	178   
8  	180   
9  	174   
10 	183   
11 	179   
12 	175   
13 	173   
14 	183   
15 	178   
16 	181   
17 	180   
18 	182   
19 	177   
20 	180   
21 	179   
22 	183   
23 	176   
24 	183   
25 	181   
26 	173   
27 	184   
28 	185   
29 	184   
30 	179   
31 	178   
32 	180   
33 	176   
34 	173   
35 	181   
36 	181   
37 	172   
38 	183   
39 	176   
40 	186   
41 	179   
42 	180   
43 	179   
44 	183   
45 	173   
46 	185   
47 	183   
48 	179   
49 	182   
50 	180   
51 	184   
52 	182   
53 	180   
54 	183   
55 	180   
56 	181   
57 	181   
58 	174   
59 	177   
60 	172   
61 	177   
62 	173   
63 	174   
64 	176   
65 	182   
66 	182   
67 	177   
68 	180   
69 	175   
70 	176   
71 	181   
72 	165   
73 	178   
74 	175   
75 	183   
76 	181   
77 	179   
78 	174   
79 	180   
80 	183   
81 	174   
82 	177   
83 	172   
84 	178   
85 	192   
86 	183   
87 	173   
88 	178   
89 	181   

745	184   
746	180   
747	176   
748	175   
749	183   
750	180   
751	184   
752	185   
753	184   
754	179   
755	180   
756	182   
757	183   
758	179   
759	184   
760	188   
761	175   
762	175   
763	183   
764	179   
765	172   
766	180   
767	184   
768	174   
769	182   
770	179   
771	176   
772	184   
773	176   
774	185   
775	180   
776	178   
777	180   
778	174   
779	180   
780	184   
781	181   
782	183   
783	175   
784	183   
785	179   
786	177   
787	179   
788	184   
789	177   
790	179   
791	182   
792	178   
793	183   
794	176   
795	185   
796	185   
797	181   
798	177   
799	181   
800	178   
801	178   
802	185   
803	184   
804	174   
805	183   
806	185   
807	179   
808	182   
809	173   
810	180   
811	185   
812	183   
813	177   
814	179   
815	181   
816	173   
817	182   
818	179   
819	171   
820	178   
821	182   
822	184   
823	179   
824	173   
825	172   
826	177   
827	183   
828	182   
829	183   
830	184   
831	181   
832	181   
833	178   
834	178   
835	189   

In [14]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B探索.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B探索.xlsx', index=False)

## Expoitation

In [15]:
pop_size = 100
n_gen = 100
CXPB = 0.5
MUTPB = 0.01

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	47    
2  	53    
3  	55    
4  	49    
5  	43    
6  	50    
7  	50    
8  	44    
9  	48    
10 	50    
11 	58    
12 	49    
13 	51    
14 	62    
15 	51    
16 	46    
17 	56    
18 	44    
19 	54    
20 	53    
21 	50    
22 	53    
23 	53    
24 	52    
25 	52    
26 	51    
27 	49    
28 	47    
29 	50    
30 	53    
31 	55    
32 	51    
33 	59    
34 	50    
35 	50    
36 	52    
37 	55    
38 	48    
39 	47    
40 	46    
41 	57    
42 	54    
43 	41    
44 	52    
45 	50    
46 	43    
47 	58    
48 	56    
49 	45    
50 	58    
51 	59    
52 	47    
53 	56    
54 	54    
55 	56    
56 	50    
57 	47    
58 	54    
59 	49    
60 	54    
61 	46    
62 	56    
63 	45    
64 	54    
65 	50    
66 	47    
67 	49    
68 	50    
69 	54    
70 	44    
71 	50    
72 	40    
73 	59    
74 	51    
75 	47    
76 	46    
77 	59    
78 	52    
79 	49    
80 	55    
81 	57    
82 	47    
83 	49    
84 	62    
85 	50    
86 	59    
87 	54    
88 	51    
89 	50    

In [16]:
# 将帕累托前沿数据保存到Excel文件
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df = pd.DataFrame(pareto_solutions, columns=column_names)
df.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [17]:
pop_size = 100
n_gen = 100
CXPB = 0.6
MUTPB = 0.01

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	61    
2  	62    
3  	63    
4  	65    
5  	62    
6  	55    
7  	56    
8  	61    
9  	66    
10 	61    
11 	58    
12 	65    
13 	60    
14 	67    
15 	60    
16 	64    
17 	62    
18 	64    
19 	62    
20 	68    
21 	64    
22 	61    
23 	61    
24 	56    
25 	60    
26 	64    
27 	64    
28 	66    
29 	60    
30 	60    
31 	64    
32 	60    
33 	58    
34 	67    
35 	67    
36 	63    
37 	57    
38 	70    
39 	68    
40 	59    
41 	56    
42 	49    
43 	55    
44 	60    
45 	65    
46 	61    
47 	63    
48 	69    
49 	61    
50 	58    
51 	65    
52 	66    
53 	67    
54 	65    
55 	59    
56 	59    
57 	66    
58 	61    
59 	59    
60 	55    
61 	56    
62 	53    
63 	71    
64 	63    
65 	58    
66 	61    
67 	49    
68 	65    
69 	59    
70 	64    
71 	62    
72 	57    
73 	63    
74 	69    
75 	60    
76 	60    
77 	56    
78 	60    
79 	58    
80 	60    
81 	57    
82 	57    
83 	59    
84 	60    
85 	70    
86 	62    
87 	63    
88 	59    
89 	55    

In [18]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [23]:
pop_size = 100
n_gen = 100
CXPB = 0.7
MUTPB = 0.01

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	72    
2  	75    
3  	72    
4  	63    
5  	72    
6  	74    
7  	66    
8  	73    
9  	63    
10 	76    
11 	82    
12 	70    
13 	66    
14 	69    
15 	68    
16 	73    
17 	73    
18 	69    
19 	67    
20 	69    
21 	73    
22 	70    
23 	70    
24 	68    
25 	72    
26 	67    
27 	72    
28 	78    
29 	73    
30 	74    
31 	73    
32 	73    
33 	76    
34 	77    
35 	82    
36 	78    
37 	74    
38 	70    
39 	77    
40 	72    
41 	63    
42 	72    
43 	71    
44 	76    
45 	71    
46 	64    
47 	64    
48 	69    
49 	70    
50 	66    
51 	72    
52 	68    
53 	66    
54 	74    
55 	74    
56 	65    
57 	70    
58 	74    
59 	71    
60 	76    
61 	67    
62 	74    
63 	67    
64 	68    
65 	72    
66 	74    
67 	71    
68 	72    
69 	70    
70 	78    
71 	78    
72 	63    
73 	79    
74 	73    
75 	67    
76 	77    
77 	70    
78 	73    
79 	76    
80 	72    
81 	73    
82 	73    
83 	75    
84 	68    
85 	67    
86 	72    
87 	74    
88 	73    
89 	67    

In [24]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [25]:
pop_size = 100
n_gen = 100
CXPB = 0.8
MUTPB = 0.01

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	77    
2  	77    
3  	86    
4  	80    
5  	81    
6  	75    
7  	75    
8  	87    
9  	77    
10 	82    
11 	80    
12 	83    
13 	79    
14 	76    
15 	79    
16 	82    
17 	84    
18 	72    
19 	71    
20 	82    
21 	83    
22 	86    
23 	83    
24 	80    
25 	84    
26 	78    
27 	83    
28 	88    
29 	79    
30 	85    
31 	86    
32 	78    
33 	80    
34 	82    
35 	73    
36 	80    
37 	84    
38 	87    
39 	85    
40 	81    
41 	84    
42 	74    
43 	85    
44 	81    
45 	82    
46 	73    
47 	78    
48 	91    
49 	82    
50 	83    
51 	82    
52 	82    
53 	76    
54 	77    
55 	82    
56 	82    
57 	72    
58 	88    
59 	80    
60 	76    
61 	73    
62 	79    
63 	74    
64 	84    
65 	78    
66 	84    
67 	88    
68 	78    
69 	86    
70 	78    
71 	81    
72 	80    
73 	83    
74 	77    
75 	81    
76 	84    
77 	76    
78 	83    
79 	82    
80 	82    
81 	74    
82 	79    
83 	87    
84 	80    
85 	84    
86 	88    
87 	79    
88 	85    
89 	77    

In [26]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [27]:
pop_size = 100
n_gen = 100
CXPB = 0.8
MUTPB = 0.02

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	81    
2  	84    
3  	85    
4  	83    
5  	80    
6  	79    
7  	89    
8  	77    
9  	86    
10 	80    
11 	79    
12 	88    
13 	85    
14 	81    
15 	82    
16 	86    
17 	76    
18 	80    
19 	84    
20 	86    
21 	83    
22 	77    
23 	79    
24 	82    
25 	87    
26 	83    
27 	87    
28 	81    
29 	80    
30 	86    
31 	87    
32 	81    
33 	74    
34 	85    
35 	85    
36 	88    
37 	92    
38 	83    
39 	78    
40 	83    
41 	83    
42 	78    
43 	80    
44 	82    
45 	78    
46 	77    
47 	79    
48 	88    
49 	85    
50 	81    
51 	87    
52 	81    
53 	83    
54 	79    
55 	76    
56 	80    
57 	85    
58 	84    
59 	84    
60 	82    
61 	82    
62 	77    
63 	85    
64 	82    
65 	79    
66 	85    
67 	91    
68 	81    
69 	90    
70 	80    
71 	80    
72 	80    
73 	83    
74 	89    
75 	85    
76 	83    
77 	82    
78 	83    
79 	77    
80 	80    
81 	86    
82 	84    
83 	79    
84 	87    
85 	84    
86 	86    
87 	79    
88 	88    
89 	87    

In [28]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [29]:
pop_size = 100
n_gen = 100
CXPB = 0.7
MUTPB = 0.02

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	62    
2  	78    
3  	66    
4  	70    
5  	68    
6  	76    
7  	70    
8  	74    
9  	76    
10 	69    
11 	74    
12 	76    
13 	70    
14 	80    
15 	76    
16 	77    
17 	72    
18 	69    
19 	71    
20 	71    
21 	76    
22 	80    
23 	70    
24 	76    
25 	69    
26 	69    
27 	67    
28 	74    
29 	70    
30 	86    
31 	78    
32 	70    
33 	77    
34 	78    
35 	67    
36 	65    
37 	79    
38 	67    
39 	73    
40 	72    
41 	67    
42 	81    
43 	77    
44 	70    
45 	68    
46 	72    
47 	66    
48 	74    
49 	67    
50 	73    
51 	68    
52 	71    
53 	71    
54 	73    
55 	72    
56 	68    
57 	72    
58 	76    
59 	73    
60 	75    
61 	56    
62 	71    
63 	77    
64 	69    
65 	68    
66 	74    
67 	65    
68 	75    
69 	78    
70 	77    
71 	66    
72 	71    
73 	72    
74 	76    
75 	74    
76 	73    
77 	67    
78 	62    
79 	63    
80 	79    
81 	67    
82 	69    
83 	78    
84 	69    
85 	75    
86 	77    
87 	71    
88 	67    
89 	71    

In [30]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [31]:
pop_size = 100
n_gen = 100
CXPB = 0.6
MUTPB = 0.02

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	56    
2  	62    
3  	61    
4  	57    
5  	64    
6  	59    
7  	68    
8  	72    
9  	61    
10 	64    
11 	60    
12 	55    
13 	68    
14 	76    
15 	61    
16 	58    
17 	64    
18 	59    
19 	65    
20 	56    
21 	64    
22 	59    
23 	68    
24 	59    
25 	68    
26 	69    
27 	62    
28 	62    
29 	60    
30 	72    
31 	67    
32 	60    
33 	60    
34 	75    
35 	59    
36 	62    
37 	64    
38 	63    
39 	56    
40 	59    
41 	70    
42 	58    
43 	63    
44 	65    
45 	67    
46 	60    
47 	58    
48 	57    
49 	68    
50 	56    
51 	65    
52 	66    
53 	70    
54 	58    
55 	60    
56 	60    
57 	61    
58 	55    
59 	55    
60 	62    
61 	56    
62 	58    
63 	67    
64 	62    
65 	62    
66 	58    
67 	63    
68 	54    
69 	59    
70 	57    
71 	70    
72 	61    
73 	67    
74 	56    
75 	50    
76 	63    
77 	62    
78 	68    
79 	65    
80 	63    
81 	62    
82 	59    
83 	61    
84 	61    
85 	68    
86 	64    
87 	68    
88 	62    
89 	55    

In [32]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [33]:
pop_size = 100
n_gen = 100
CXPB = 0.5
MUTPB = 0.02

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	55    
2  	47    
3  	46    
4  	59    
5  	64    
6  	56    
7  	53    
8  	50    
9  	53    
10 	51    
11 	48    
12 	60    
13 	58    
14 	52    
15 	51    
16 	50    
17 	48    
18 	54    
19 	56    
20 	48    
21 	48    
22 	51    
23 	52    
24 	51    
25 	59    
26 	54    
27 	57    
28 	53    
29 	38    
30 	55    
31 	57    
32 	49    
33 	46    
34 	53    
35 	52    
36 	54    
37 	47    
38 	48    
39 	50    
40 	54    
41 	46    
42 	56    
43 	59    
44 	46    
45 	56    
46 	47    
47 	46    
48 	57    
49 	46    
50 	54    
51 	41    
52 	54    
53 	48    
54 	63    
55 	59    
56 	47    
57 	48    
58 	55    
59 	55    
60 	53    
61 	48    
62 	57    
63 	48    
64 	49    
65 	47    
66 	56    
67 	38    
68 	45    
69 	44    
70 	53    
71 	56    
72 	57    
73 	46    
74 	58    
75 	43    
76 	50    
77 	53    
78 	54    
79 	51    
80 	54    
81 	65    
82 	56    
83 	56    
84 	52    
85 	53    
86 	47    
87 	50    
88 	53    
89 	50    

In [34]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [35]:
pop_size = 100
n_gen = 100
CXPB = 0.5
MUTPB = 0.03

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	53    
2  	57    
3  	50    
4  	53    
5  	48    
6  	51    
7  	56    
8  	51    
9  	59    
10 	45    
11 	56    
12 	56    
13 	48    
14 	56    
15 	51    
16 	53    
17 	45    
18 	62    
19 	56    
20 	49    
21 	53    
22 	56    
23 	62    
24 	58    
25 	44    
26 	55    
27 	63    
28 	55    
29 	57    
30 	48    
31 	45    
32 	50    
33 	54    
34 	54    
35 	48    
36 	52    
37 	54    
38 	60    
39 	60    
40 	49    
41 	54    
42 	50    
43 	42    
44 	55    
45 	49    
46 	53    
47 	45    
48 	53    
49 	43    
50 	52    
51 	51    
52 	56    
53 	50    
54 	46    
55 	46    
56 	54    
57 	56    
58 	53    
59 	55    
60 	49    
61 	53    
62 	60    
63 	42    
64 	54    
65 	59    
66 	43    
67 	54    
68 	57    
69 	51    
70 	60    
71 	47    
72 	47    
73 	64    
74 	53    
75 	60    
76 	51    
77 	52    
78 	56    
79 	54    
80 	45    
81 	53    
82 	51    
83 	52    
84 	49    
85 	51    
86 	53    
87 	51    
88 	46    
89 	53    

In [36]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [37]:
pop_size = 100
n_gen = 100
CXPB = 0.6
MUTPB = 0.03

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	54    
2  	62    
3  	64    
4  	67    
5  	63    
6  	62    
7  	59    
8  	60    
9  	64    
10 	59    
11 	73    
12 	63    
13 	68    
14 	59    
15 	61    
16 	60    
17 	61    
18 	67    
19 	65    
20 	63    
21 	64    
22 	57    
23 	67    
24 	56    
25 	64    
26 	62    
27 	62    
28 	66    
29 	67    
30 	59    
31 	71    
32 	65    
33 	65    
34 	62    
35 	54    
36 	66    
37 	69    
38 	67    
39 	65    
40 	68    
41 	64    
42 	60    
43 	65    
44 	64    
45 	58    
46 	70    
47 	58    
48 	66    
49 	64    
50 	68    
51 	64    
52 	69    
53 	63    
54 	65    
55 	58    
56 	56    
57 	66    
58 	63    
59 	62    
60 	64    
61 	58    
62 	63    
63 	60    
64 	64    
65 	67    
66 	54    
67 	63    
68 	68    
69 	62    
70 	55    
71 	65    
72 	61    
73 	59    
74 	54    
75 	57    
76 	52    
77 	60    
78 	64    
79 	64    
80 	61    
81 	55    
82 	64    
83 	55    
84 	67    
85 	52    
86 	61    
87 	61    
88 	61    
89 	65    

In [38]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [39]:
pop_size = 100
n_gen = 100
CXPB = 0.7
MUTPB = 0.03

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	73    
2  	74    
3  	71    
4  	77    
5  	70    
6  	67    
7  	82    
8  	71    
9  	75    
10 	69    
11 	77    
12 	79    
13 	69    
14 	70    
15 	71    
16 	70    
17 	70    
18 	72    
19 	78    
20 	69    
21 	74    
22 	74    
23 	78    
24 	68    
25 	77    
26 	72    
27 	78    
28 	78    
29 	77    
30 	68    
31 	65    
32 	68    
33 	76    
34 	77    
35 	76    
36 	65    
37 	70    
38 	74    
39 	75    
40 	69    
41 	70    
42 	70    
43 	69    
44 	70    
45 	69    
46 	75    
47 	77    
48 	77    
49 	85    
50 	77    
51 	76    
52 	74    
53 	74    
54 	64    
55 	81    
56 	73    
57 	64    
58 	80    
59 	67    
60 	71    
61 	73    
62 	71    
63 	74    
64 	74    
65 	74    
66 	68    
67 	71    
68 	70    
69 	75    
70 	76    
71 	70    
72 	65    
73 	69    
74 	77    
75 	73    
76 	76    
77 	77    
78 	71    
79 	71    
80 	71    
81 	62    
82 	67    
83 	77    
84 	71    
85 	72    
86 	66    
87 	79    
88 	72    
89 	71    

In [40]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [41]:
pop_size = 100
n_gen = 100
CXPB = 0.8
MUTPB = 0.03

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	85    
2  	85    
3  	87    
4  	83    
5  	82    
6  	88    
7  	85    
8  	82    
9  	86    
10 	81    
11 	84    
12 	83    
13 	85    
14 	85    
15 	90    
16 	80    
17 	79    
18 	77    
19 	84    
20 	74    
21 	86    
22 	85    
23 	82    
24 	85    
25 	79    
26 	82    
27 	86    
28 	90    
29 	89    
30 	85    
31 	87    
32 	89    
33 	88    
34 	77    
35 	89    
36 	80    
37 	79    
38 	88    
39 	83    
40 	91    
41 	82    
42 	80    
43 	83    
44 	82    
45 	86    
46 	82    
47 	80    
48 	85    
49 	81    
50 	83    
51 	83    
52 	85    
53 	84    
54 	84    
55 	89    
56 	86    
57 	82    
58 	84    
59 	75    
60 	85    
61 	87    
62 	84    
63 	85    
64 	78    
65 	93    
66 	86    
67 	90    
68 	81    
69 	86    
70 	83    
71 	80    
72 	86    
73 	81    
74 	79    
75 	84    
76 	85    
77 	80    
78 	84    
79 	79    
80 	84    
81 	80    
82 	80    
83 	86    
84 	83    
85 	84    
86 	90    
87 	83    
88 	84    
89 	83    

In [42]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [43]:
pop_size = 100
n_gen = 100
CXPB = 0.8
MUTPB = 0.04

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	86    
2  	80    
3  	84    
4  	89    
5  	84    
6  	81    
7  	85    
8  	84    
9  	85    
10 	85    
11 	82    
12 	89    
13 	78    
14 	83    
15 	84    
16 	84    
17 	85    
18 	89    
19 	77    
20 	84    
21 	82    
22 	88    
23 	83    
24 	91    
25 	84    
26 	88    
27 	87    
28 	82    
29 	82    
30 	92    
31 	78    
32 	87    
33 	81    
34 	84    
35 	82    
36 	82    
37 	79    
38 	90    
39 	85    
40 	85    
41 	80    
42 	92    
43 	88    
44 	88    
45 	78    
46 	84    
47 	80    
48 	87    
49 	83    
50 	91    
51 	87    
52 	87    
53 	85    
54 	90    
55 	76    
56 	82    
57 	81    
58 	82    
59 	84    
60 	83    
61 	84    
62 	90    
63 	87    
64 	80    
65 	86    
66 	81    
67 	79    
68 	85    
69 	93    
70 	85    
71 	82    
72 	86    
73 	81    
74 	87    
75 	81    
76 	81    
77 	90    
78 	87    
79 	82    
80 	82    
81 	83    
82 	86    
83 	88    
84 	80    
85 	79    
86 	82    
87 	79    
88 	82    
89 	86    

In [44]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [45]:
pop_size = 100
n_gen = 100
CXPB = 0.7
MUTPB = 0.04

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	76    
2  	70    
3  	72    
4  	75    
5  	69    
6  	73    
7  	69    
8  	81    
9  	78    
10 	82    
11 	74    
12 	75    
13 	79    
14 	74    
15 	76    
16 	67    
17 	76    
18 	75    
19 	74    
20 	69    
21 	70    
22 	74    
23 	76    
24 	76    
25 	72    
26 	79    
27 	74    
28 	73    
29 	65    
30 	69    
31 	71    
32 	71    
33 	78    
34 	64    
35 	71    
36 	73    
37 	79    
38 	76    
39 	77    
40 	72    
41 	72    
42 	78    
43 	69    
44 	74    
45 	72    
46 	76    
47 	79    
48 	78    
49 	69    
50 	71    
51 	73    
52 	72    
53 	67    
54 	73    
55 	72    
56 	71    
57 	72    
58 	74    
59 	72    
60 	79    
61 	74    
62 	75    
63 	64    
64 	80    
65 	73    
66 	77    
67 	72    
68 	76    
69 	72    
70 	70    
71 	76    
72 	69    
73 	74    
74 	75    
75 	68    
76 	73    
77 	74    
78 	75    
79 	66    
80 	77    
81 	68    
82 	67    
83 	75    
84 	70    
85 	73    
86 	71    
87 	74    
88 	75    
89 	74    

In [46]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [47]:
pop_size = 100
n_gen = 100
CXPB = 0.6
MUTPB = 0.04

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	71    
2  	67    
3  	61    
4  	68    
5  	65    
6  	64    
7  	66    
8  	56    
9  	50    
10 	60    
11 	60    
12 	63    
13 	62    
14 	70    
15 	64    
16 	73    
17 	56    
18 	60    
19 	66    
20 	55    
21 	59    
22 	63    
23 	67    
24 	67    
25 	59    
26 	67    
27 	72    
28 	69    
29 	58    
30 	68    
31 	64    
32 	64    
33 	65    
34 	63    
35 	64    
36 	65    
37 	58    
38 	62    
39 	69    
40 	62    
41 	68    
42 	61    
43 	67    
44 	65    
45 	63    
46 	60    
47 	61    
48 	68    
49 	59    
50 	63    
51 	61    
52 	61    
53 	68    
54 	57    
55 	66    
56 	62    
57 	67    
58 	63    
59 	69    
60 	56    
61 	60    
62 	63    
63 	72    
64 	74    
65 	64    
66 	51    
67 	64    
68 	61    
69 	66    
70 	61    
71 	61    
72 	63    
73 	64    
74 	68    
75 	74    
76 	66    
77 	50    
78 	68    
79 	59    
80 	62    
81 	65    
82 	69    
83 	71    
84 	63    
85 	73    
86 	62    
87 	67    
88 	58    
89 	69    

In [48]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [49]:
pop_size = 100
n_gen = 100
CXPB = 0.5
MUTPB = 0.04

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	57    
2  	60    
3  	56    
4  	51    
5  	52    
6  	55    
7  	55    
8  	53    
9  	48    
10 	53    
11 	59    
12 	59    
13 	51    
14 	55    
15 	47    
16 	58    
17 	56    
18 	53    
19 	48    
20 	59    
21 	61    
22 	48    
23 	48    
24 	52    
25 	53    
26 	46    
27 	55    
28 	50    
29 	56    
30 	53    
31 	59    
32 	64    
33 	66    
34 	53    
35 	48    
36 	40    
37 	50    
38 	49    
39 	61    
40 	59    
41 	56    
42 	48    
43 	44    
44 	67    
45 	50    
46 	50    
47 	56    
48 	52    
49 	63    
50 	53    
51 	59    
52 	56    
53 	50    
54 	48    
55 	50    
56 	54    
57 	55    
58 	50    
59 	56    
60 	58    
61 	61    
62 	48    
63 	55    
64 	64    
65 	55    
66 	50    
67 	41    
68 	64    
69 	59    
70 	50    
71 	61    
72 	52    
73 	55    
74 	57    
75 	48    
76 	48    
77 	56    
78 	53    
79 	54    
80 	43    
81 	55    
82 	51    
83 	45    
84 	55    
85 	59    
86 	49    
87 	51    
88 	53    
89 	51    

In [50]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [51]:
pop_size = 100
n_gen = 100
CXPB = 0.5
MUTPB = 0.05

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	56    
2  	55    
3  	55    
4  	52    
5  	55    
6  	63    
7  	54    
8  	60    
9  	59    
10 	53    
11 	53    
12 	57    
13 	50    
14 	59    
15 	61    
16 	59    
17 	58    
18 	51    
19 	50    
20 	56    
21 	51    
22 	54    
23 	55    
24 	60    
25 	56    
26 	55    
27 	53    
28 	43    
29 	48    
30 	57    
31 	53    
32 	58    
33 	50    
34 	56    
35 	64    
36 	54    
37 	52    
38 	48    
39 	56    
40 	54    
41 	59    
42 	47    
43 	60    
44 	56    
45 	53    
46 	61    
47 	51    
48 	55    
49 	65    
50 	54    
51 	49    
52 	56    
53 	52    
54 	54    
55 	56    
56 	54    
57 	56    
58 	53    
59 	56    
60 	54    
61 	54    
62 	52    
63 	57    
64 	49    
65 	53    
66 	48    
67 	53    
68 	47    
69 	54    
70 	60    
71 	58    
72 	60    
73 	62    
74 	57    
75 	57    
76 	63    
77 	46    
78 	64    
79 	56    
80 	56    
81 	45    
82 	57    
83 	53    
84 	49    
85 	53    
86 	48    
87 	58    
88 	53    
89 	66    

In [52]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [54]:
pop_size = 100
n_gen = 100
CXPB = 0.6
MUTPB = 0.05

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	72    
2  	66    
3  	57    
4  	68    
5  	59    
6  	64    
7  	68    
8  	55    
9  	71    
10 	64    
11 	68    
12 	69    
13 	60    
14 	62    
15 	64    
16 	69    
17 	67    
18 	62    
19 	52    
20 	61    
21 	67    
22 	64    
23 	56    
24 	67    
25 	58    
26 	68    
27 	65    
28 	61    
29 	71    
30 	62    
31 	66    
32 	64    
33 	66    
34 	64    
35 	67    
36 	56    
37 	58    
38 	75    
39 	66    
40 	58    
41 	60    
42 	68    
43 	62    
44 	70    
45 	59    
46 	65    
47 	74    
48 	62    
49 	65    
50 	62    
51 	67    
52 	54    
53 	67    
54 	64    
55 	60    
56 	63    
57 	63    
58 	62    
59 	66    
60 	67    
61 	67    
62 	65    
63 	68    
64 	64    
65 	54    
66 	62    
67 	64    
68 	67    
69 	67    
70 	62    
71 	65    
72 	69    
73 	68    
74 	66    
75 	66    
76 	72    
77 	69    
78 	68    
79 	68    
80 	67    
81 	63    
82 	69    
83 	58    
84 	66    
85 	67    
86 	69    
87 	66    
88 	68    
89 	62    

In [55]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [56]:
pop_size = 100
n_gen = 100
CXPB = 0.7
MUTPB = 0.05

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	75    
2  	78    
3  	68    
4  	74    
5  	69    
6  	72    
7  	72    
8  	78    
9  	72    
10 	74    
11 	82    
12 	73    
13 	74    
14 	74    
15 	76    
16 	68    
17 	70    
18 	73    
19 	76    
20 	73    
21 	76    
22 	73    
23 	72    
24 	67    
25 	74    
26 	74    
27 	86    
28 	85    
29 	79    
30 	68    
31 	75    
32 	77    
33 	72    
34 	72    
35 	80    
36 	67    
37 	84    
38 	75    
39 	80    
40 	65    
41 	72    
42 	75    
43 	79    
44 	81    
45 	68    
46 	74    
47 	72    
48 	79    
49 	80    
50 	80    
51 	74    
52 	81    
53 	72    
54 	81    
55 	76    
56 	73    
57 	76    
58 	69    
59 	80    
60 	77    
61 	70    
62 	73    
63 	72    
64 	80    
65 	77    
66 	79    
67 	83    
68 	80    
69 	75    
70 	78    
71 	78    
72 	75    
73 	83    
74 	71    
75 	72    
76 	68    
77 	71    
78 	71    
79 	78    
80 	66    
81 	82    
82 	82    
83 	74    
84 	74    
85 	80    
86 	74    
87 	77    
88 	76    
89 	75    

In [57]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

In [58]:
pop_size = 100
n_gen = 100
CXPB = 0.8
MUTPB = 0.05

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	82    
2  	90    
3  	78    
4  	83    
5  	83    
6  	90    
7  	84    
8  	89    
9  	86    
10 	87    
11 	82    
12 	92    
13 	86    
14 	81    
15 	86    
16 	88    
17 	87    
18 	78    
19 	78    
20 	87    
21 	85    
22 	87    
23 	87    
24 	89    
25 	86    
26 	80    
27 	84    
28 	88    
29 	86    
30 	86    
31 	82    
32 	85    
33 	89    
34 	89    
35 	89    
36 	87    
37 	91    
38 	87    
39 	83    
40 	88    
41 	80    
42 	84    
43 	76    
44 	85    
45 	87    
46 	82    
47 	84    
48 	92    
49 	81    
50 	85    
51 	84    
52 	88    
53 	84    
54 	86    
55 	82    
56 	82    
57 	84    
58 	89    
59 	85    
60 	88    
61 	81    
62 	84    
63 	82    
64 	88    
65 	84    
66 	88    
67 	90    
68 	86    
69 	78    
70 	91    
71 	86    
72 	85    
73 	83    
74 	85    
75 	87    
76 	81    
77 	90    
78 	85    
79 	90    
80 	89    
81 	89    
82 	86    
83 	81    
84 	87    
85 	83    
86 	80    
87 	87    
88 	86    
89 	91    

In [59]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B利用.xlsx', index=False)

## Balance

In [60]:
pop_size = 100
n_gen = 500
CXPB = 0.6
MUTPB = 0.03

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	59    
2  	58    
3  	60    
4  	66    
5  	63    
6  	62    
7  	60    
8  	74    
9  	64    
10 	62    
11 	67    
12 	70    
13 	62    
14 	63    
15 	62    
16 	63    
17 	66    
18 	67    
19 	71    
20 	56    
21 	68    
22 	63    
23 	69    
24 	62    
25 	61    
26 	64    
27 	63    
28 	59    
29 	59    
30 	60    
31 	60    
32 	65    
33 	61    
34 	63    
35 	68    
36 	70    
37 	60    
38 	65    
39 	60    
40 	61    
41 	67    
42 	59    
43 	71    
44 	60    
45 	55    
46 	58    
47 	59    
48 	70    
49 	64    
50 	65    
51 	56    
52 	59    
53 	64    
54 	63    
55 	64    
56 	58    
57 	64    
58 	63    
59 	75    
60 	63    
61 	68    
62 	60    
63 	76    
64 	66    
65 	65    
66 	61    
67 	69    
68 	65    
69 	64    
70 	63    
71 	66    
72 	67    
73 	71    
74 	67    
75 	54    
76 	66    
77 	55    
78 	54    
79 	59    
80 	57    
81 	62    
82 	69    
83 	63    
84 	61    
85 	63    
86 	69    
87 	70    
88 	63    
89 	61    

In [61]:
# 将帕累托前沿数据保存到Excel文件
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df = pd.DataFrame(pareto_solutions, columns=column_names)
df.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx', index=False)

In [62]:
pop_size = 100
n_gen = 500
CXPB = 0.7
MUTPB = 0.03

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	69    
2  	78    
3  	77    
4  	75    
5  	73    
6  	68    
7  	71    
8  	73    
9  	75    
10 	79    
11 	68    
12 	72    
13 	71    
14 	75    
15 	66    
16 	68    
17 	74    
18 	68    
19 	67    
20 	69    
21 	72    
22 	74    
23 	65    
24 	71    
25 	73    
26 	78    
27 	74    
28 	63    
29 	69    
30 	69    
31 	69    
32 	74    
33 	68    
34 	77    
35 	70    
36 	82    
37 	70    
38 	74    
39 	81    
40 	66    
41 	69    
42 	73    
43 	73    
44 	68    
45 	74    
46 	66    
47 	68    
48 	70    
49 	78    
50 	73    
51 	74    
52 	72    
53 	75    
54 	75    
55 	71    
56 	76    
57 	79    
58 	73    
59 	72    
60 	72    
61 	68    
62 	78    
63 	83    
64 	71    
65 	72    
66 	80    
67 	67    
68 	78    
69 	70    
70 	73    
71 	73    
72 	69    
73 	74    
74 	65    
75 	70    
76 	74    
77 	76    
78 	80    
79 	70    
80 	71    
81 	74    
82 	78    
83 	67    
84 	64    
85 	69    
86 	75    
87 	72    
88 	70    
89 	76    

In [63]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx', index=False)

In [64]:
pop_size = 100
n_gen = 500
CXPB = 0.7
MUTPB = 0.04

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	79    
2  	76    
3  	77    
4  	73    
5  	74    
6  	73    
7  	80    
8  	71    
9  	75    
10 	76    
11 	70    
12 	70    
13 	82    
14 	84    
15 	76    
16 	81    
17 	70    
18 	74    
19 	71    
20 	76    
21 	75    
22 	72    
23 	69    
24 	83    
25 	67    
26 	74    
27 	74    
28 	75    
29 	83    
30 	76    
31 	80    
32 	77    
33 	82    
34 	68    
35 	73    
36 	79    
37 	78    
38 	79    
39 	68    
40 	75    
41 	75    
42 	72    
43 	72    
44 	88    
45 	76    
46 	78    
47 	72    
48 	71    
49 	77    
50 	73    
51 	68    
52 	74    
53 	69    
54 	72    
55 	70    
56 	71    
57 	77    
58 	78    
59 	78    
60 	83    
61 	76    
62 	79    
63 	77    
64 	79    
65 	79    
66 	74    
67 	68    
68 	75    
69 	70    
70 	68    
71 	71    
72 	75    
73 	74    
74 	77    
75 	67    
76 	73    
77 	75    
78 	70    
79 	75    
80 	82    
81 	73    
82 	71    
83 	71    
84 	76    
85 	71    
86 	76    
87 	77    
88 	81    
89 	76    

In [65]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx', index=False)

In [66]:
pop_size = 100
n_gen = 500
CXPB = 0.6
MUTPB = 0.04

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	61    
2  	65    
3  	66    
4  	50    
5  	66    
6  	64    
7  	64    
8  	58    
9  	61    
10 	65    
11 	63    
12 	54    
13 	71    
14 	63    
15 	67    
16 	60    
17 	61    
18 	60    
19 	66    
20 	60    
21 	70    
22 	63    
23 	55    
24 	61    
25 	62    
26 	69    
27 	62    
28 	66    
29 	60    
30 	62    
31 	63    
32 	73    
33 	60    
34 	64    
35 	66    
36 	62    
37 	65    
38 	61    
39 	67    
40 	60    
41 	60    
42 	65    
43 	67    
44 	66    
45 	62    
46 	59    
47 	66    
48 	68    
49 	67    
50 	69    
51 	57    
52 	65    
53 	63    
54 	63    
55 	62    
56 	53    
57 	59    
58 	53    
59 	63    
60 	73    
61 	65    
62 	68    
63 	64    
64 	68    
65 	66    
66 	69    
67 	70    
68 	63    
69 	72    
70 	62    
71 	61    
72 	62    
73 	61    
74 	63    
75 	56    
76 	68    
77 	67    
78 	59    
79 	64    
80 	62    
81 	56    
82 	58    
83 	75    
84 	60    
85 	54    
86 	66    
87 	63    
88 	67    
89 	60    

In [67]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx', index=False)

In [68]:
pop_size = 100
n_gen = 500
CXPB = 0.6
MUTPB = 0.05

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	68    
2  	66    
3  	60    
4  	75    
5  	60    
6  	64    
7  	70    
8  	66    
9  	66    
10 	65    
11 	71    
12 	68    
13 	70    
14 	58    
15 	68    
16 	68    
17 	68    
18 	62    
19 	64    
20 	62    
21 	75    
22 	66    
23 	52    
24 	61    
25 	64    
26 	62    
27 	59    
28 	66    
29 	58    
30 	63    
31 	65    
32 	67    
33 	64    
34 	72    
35 	63    
36 	71    
37 	60    
38 	68    
39 	62    
40 	64    
41 	66    
42 	70    
43 	70    
44 	65    
45 	57    
46 	63    
47 	62    
48 	65    
49 	65    
50 	66    
51 	62    
52 	73    
53 	68    
54 	72    
55 	65    
56 	65    
57 	60    
58 	64    
59 	64    
60 	62    
61 	63    
62 	69    
63 	68    
64 	63    
65 	68    
66 	71    
67 	64    
68 	65    
69 	69    
70 	57    
71 	67    
72 	66    
73 	63    
74 	74    
75 	64    
76 	66    
77 	69    
78 	53    
79 	63    
80 	64    
81 	60    
82 	61    
83 	65    
84 	68    
85 	74    
86 	67    
87 	67    
88 	68    
89 	63    

In [69]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx', index=False)

In [70]:
pop_size = 100
n_gen = 500
CXPB = 0.7
MUTPB = 0.05

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	77    
2  	73    
3  	71    
4  	74    
5  	78    
6  	81    
7  	76    
8  	69    
9  	78    
10 	65    
11 	78    
12 	74    
13 	76    
14 	79    
15 	72    
16 	77    
17 	69    
18 	73    
19 	77    
20 	78    
21 	74    
22 	72    
23 	67    
24 	77    
25 	76    
26 	79    
27 	84    
28 	85    
29 	77    
30 	77    
31 	74    
32 	71    
33 	78    
34 	78    
35 	77    
36 	81    
37 	76    
38 	62    
39 	76    
40 	75    
41 	78    
42 	77    
43 	69    
44 	76    
45 	73    
46 	86    
47 	73    
48 	79    
49 	67    
50 	71    
51 	68    
52 	77    
53 	77    
54 	70    
55 	74    
56 	66    
57 	75    
58 	74    
59 	73    
60 	73    
61 	74    
62 	70    
63 	70    
64 	77    
65 	81    
66 	77    
67 	81    
68 	65    
69 	76    
70 	73    
71 	73    
72 	72    
73 	76    
74 	82    
75 	76    
76 	84    
77 	77    
78 	76    
79 	72    
80 	80    
81 	71    
82 	74    
83 	72    
84 	76    
85 	79    
86 	73    
87 	76    
88 	74    
89 	85    

In [73]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx', index=False)

In [72]:
pop_size = 100
n_gen = 500
CXPB = 0.7
MUTPB = 0.06

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	82    
2  	72    
3  	72    
4  	73    
5  	71    
6  	66    
7  	74    
8  	81    
9  	75    
10 	79    
11 	71    
12 	76    
13 	83    
14 	74    
15 	71    
16 	70    
17 	77    
18 	77    
19 	75    
20 	78    
21 	78    
22 	82    
23 	63    
24 	76    
25 	74    
26 	72    
27 	70    
28 	81    
29 	71    
30 	75    
31 	70    
32 	74    
33 	76    
34 	85    
35 	72    
36 	73    
37 	80    
38 	72    
39 	66    
40 	75    
41 	74    
42 	76    
43 	78    
44 	82    
45 	77    
46 	76    
47 	83    
48 	72    
49 	72    
50 	83    
51 	76    
52 	79    
53 	75    
54 	79    
55 	86    
56 	76    
57 	69    
58 	72    
59 	74    
60 	78    
61 	75    
62 	78    
63 	76    
64 	80    
65 	74    
66 	81    
67 	70    
68 	78    
69 	66    
70 	76    
71 	70    
72 	75    
73 	79    
74 	74    
75 	67    
76 	72    
77 	73    
78 	77    
79 	79    
80 	72    
81 	78    
82 	69    
83 	69    
84 	71    
85 	74    
86 	71    
87 	74    
88 	73    
89 	70    

In [74]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx', index=False)

In [75]:
pop_size = 100
n_gen = 500
CXPB = 0.6
MUTPB = 0.06

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	60    
2  	65    
3  	67    
4  	76    
5  	70    
6  	66    
7  	71    
8  	70    
9  	70    
10 	71    
11 	70    
12 	62    
13 	64    
14 	66    
15 	66    
16 	55    
17 	71    
18 	66    
19 	61    
20 	68    
21 	71    
22 	63    
23 	69    
24 	72    
25 	65    
26 	64    
27 	61    
28 	64    
29 	74    
30 	66    
31 	59    
32 	65    
33 	65    
34 	73    
35 	67    
36 	66    
37 	73    
38 	65    
39 	68    
40 	61    
41 	69    
42 	70    
43 	67    
44 	69    
45 	65    
46 	59    
47 	60    
48 	63    
49 	66    
50 	63    
51 	61    
52 	60    
53 	68    
54 	68    
55 	66    
56 	58    
57 	73    
58 	65    
59 	60    
60 	65    
61 	74    
62 	64    
63 	72    
64 	64    
65 	68    
66 	63    
67 	61    
68 	69    
69 	67    
70 	62    
71 	56    
72 	67    
73 	59    
74 	68    
75 	69    
76 	70    
77 	68    
78 	74    
79 	68    
80 	67    
81 	66    
82 	60    
83 	62    
84 	70    
85 	60    
86 	65    
87 	60    
88 	65    
89 	70    

In [76]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx', index=False)

In [77]:
pop_size = 100
n_gen = 500
CXPB = 0.7
MUTPB = 0.06

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	80    
2  	74    
3  	66    
4  	70    
5  	79    
6  	78    
7  	78    
8  	78    
9  	77    
10 	77    
11 	78    
12 	83    
13 	77    
14 	77    
15 	75    
16 	68    
17 	72    
18 	74    
19 	80    
20 	76    
21 	76    
22 	73    
23 	77    
24 	84    
25 	82    
26 	74    
27 	76    
28 	77    
29 	76    
30 	68    
31 	77    
32 	73    
33 	78    
34 	70    
35 	73    
36 	75    
37 	83    
38 	79    
39 	81    
40 	81    
41 	84    
42 	76    
43 	78    
44 	79    
45 	76    
46 	72    
47 	75    
48 	77    
49 	77    
50 	77    
51 	75    
52 	74    
53 	74    
54 	72    
55 	84    
56 	72    
57 	80    
58 	80    
59 	65    
60 	82    
61 	76    
62 	77    
63 	78    
64 	74    
65 	72    
66 	81    
67 	80    
68 	72    
69 	77    
70 	64    
71 	74    
72 	81    
73 	74    
74 	73    
75 	75    
76 	79    
77 	78    
78 	79    
79 	74    
80 	78    
81 	75    
82 	77    
83 	81    
84 	80    
85 	70    
86 	74    
87 	76    
88 	77    
89 	75    

In [78]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx', index=False)

In [79]:
pop_size = 100
n_gen = 500
CXPB = 0.7
MUTPB = 0.07

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	72    
2  	69    
3  	73    
4  	64    
5  	77    
6  	80    
7  	79    
8  	83    
9  	83    
10 	83    
11 	78    
12 	85    
13 	76    
14 	76    
15 	77    
16 	76    
17 	74    
18 	78    
19 	74    
20 	68    
21 	75    
22 	78    
23 	70    
24 	78    
25 	76    
26 	78    
27 	75    
28 	85    
29 	74    
30 	76    
31 	76    
32 	78    
33 	74    
34 	78    
35 	77    
36 	78    
37 	82    
38 	72    
39 	85    
40 	81    
41 	77    
42 	77    
43 	85    
44 	73    
45 	78    
46 	74    
47 	73    
48 	75    
49 	80    
50 	83    
51 	81    
52 	77    
53 	83    
54 	73    
55 	80    
56 	73    
57 	61    
58 	74    
59 	79    
60 	77    
61 	81    
62 	78    
63 	74    
64 	81    
65 	75    
66 	73    
67 	74    
68 	79    
69 	78    
70 	82    
71 	71    
72 	78    
73 	69    
74 	80    
75 	76    
76 	76    
77 	78    
78 	81    
79 	77    
80 	77    
81 	69    
82 	77    
83 	77    
84 	78    
85 	86    
86 	75    
87 	84    
88 	82    
89 	76    

In [80]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx', index=False)

In [81]:
pop_size = 100
n_gen = 500
CXPB = 0.6
MUTPB = 0.07

population = toolbox.population(n=pop_size)

# 使用鲸鱼优化算法演化种群
population, logbook = algorithms.eaMuPlusLambda(population, toolbox, mu=pop_size, lambda_=pop_size, cxpb=CXPB, mutpb=MUTPB, ngen=n_gen)

# 获取帕累托前沿
fronts = tools.sortNondominated(population, len(population), first_front_only=True)

# 打印 Pareto 前沿
print("Pareto front:")
for ind in fronts[0]:
    print(f'Composition: {ind},  values: {ind.fitness.values}')

gen	nevals
0  	100   
1  	65    
2  	65    
3  	75    
4  	66    
5  	74    
6  	70    
7  	67    
8  	70    
9  	64    
10 	70    
11 	70    
12 	62    
13 	72    
14 	65    
15 	61    
16 	69    
17 	72    
18 	58    
19 	68    
20 	67    
21 	79    
22 	69    
23 	67    
24 	68    
25 	62    
26 	59    
27 	76    
28 	70    
29 	65    
30 	78    
31 	72    
32 	66    
33 	63    
34 	62    
35 	56    
36 	69    
37 	65    
38 	63    
39 	69    
40 	74    
41 	68    
42 	75    
43 	64    
44 	62    
45 	64    
46 	67    
47 	70    
48 	53    
49 	71    
50 	70    
51 	62    
52 	66    
53 	70    
54 	65    
55 	63    
56 	68    
57 	69    
58 	68    
59 	72    
60 	68    
61 	71    
62 	64    
63 	72    
64 	58    
65 	60    
66 	68    
67 	67    
68 	66    
69 	80    
70 	60    
71 	67    
72 	67    
73 	66    
74 	65    
75 	64    
76 	75    
77 	62    
78 	66    
79 	63    
80 	62    
81 	74    
82 	67    
83 	62    
84 	70    
85 	66    
86 	68    
87 	63    
88 	73    
89 	63    

In [82]:
# 读取原始数据
df_original = pd.read_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx')

# 将帕累托前沿数据保存到DataFrame对象
pareto_solutions = []
for ind in fronts[0]:
    row = list(ind)  # 添加composition
    row.extend(ind.fitness.values)  # 添加values
    pareto_solutions.append(row)

column_names = element_names[:len(fronts[0])] + ["TR_Value", "TS_Value"]
df_new = pd.DataFrame(pareto_solutions, columns=column_names)

# 将原始数据和新数据合并
df_combined = pd.concat([df_original, df_new], ignore_index=True)

# 保存合并后的数据到同一个Excel文件
df_combined.to_excel(r'D:\creep rupture\添加高温测试和高应力\NSWO-B平衡.xlsx', index=False)